# Coffee17 Focused C1/C2 Screening V1

Train **only C1 and C2** over the five frozen Coffee17 development folds. Historical B0 from the completed master-screening analysis package is verified fold-by-fold and is **never retrained**. Outer test remains locked.


In [ ]:
NOTEBOOK_BUILD = "FOCUSED-C1C2-V1"
NOTEBOOK_RUNTIME_REV = "HISTORICAL-B0-VERIFIED-V1-20261007"
SCIENTIFIC_CODE_COMMIT = "6653a50e296f7b38591100b395f1a09dbaf68c56"

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-focused-c1c2-v1"
PROJECT = WORK / "coffee17-focused-c1c2-v1"
OUT = PROJECT / "experiments"
CONFIG_REL = "configs/frequency_screening/FOCUSED_EFFICIENCY_V1.yaml"
CANDIDATES = ["C1", "C2"]

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("NOTEBOOK_RUNTIME_REV:", NOTEBOOK_RUNTIME_REV)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("TRAINING CANDIDATES:", CANDIDATES)
print("B0 TRAINING: DISABLED — VERIFIED HISTORICAL ANCHOR ONLY")
print("OUTER TEST: LOCKED")

assert INPUT.is_dir() and WORK.is_dir(), "Notebook harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)

run(
    ["git", "clone", "--quiet", "https://github.com/ediprin/coffee-bean-classification.git", REPO],
    cwd=WORK,
)
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q", "-r", REPO / "requirements/preprocessing-study.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum training C1/C2.")
print("GPU:", torch.cuda.get_device_name(0))
print("DETERMINISM_MODE: SEEDED_MATCHED (strict CUDA determinism OFF)")

run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_frequency_screening.py",
    "tests/test_focused_c1c2.py",
], cwd=REPO)
print("FOCUSED TESTS: PASS")


In [ ]:
# ------------------------------------------------------------
# Historical master-screening analysis package
# ------------------------------------------------------------
HIST_WORK = WORK / "_historical_master_analysis"
shutil.rmtree(HIST_WORK, ignore_errors=True)

package_candidates = sorted(
    p for p in INPUT.rglob("coffee17-master-screening-analysis-package.zip")
    if p.is_file()
)

if len(package_candidates) == 1:
    HIST_PACKAGE = package_candidates[0]
    HIST_WORK.mkdir(parents=True)
    with zipfile.ZipFile(HIST_PACKAGE) as zf:
        zf.extractall(HIST_WORK)
    HIST_ROOT = HIST_WORK
    print("HISTORICAL PACKAGE:", HIST_PACKAGE)
else:
    # Support an already-extracted attached output, but require exactly one
    # master summary so we never guess between multiple histories.
    summaries = sorted(
        p for p in INPUT.rglob("master_screening_v1.json")
        if p.is_file()
    )
    if len(package_candidates) > 1:
        raise RuntimeError(
            "Lebih dari satu historical analysis ZIP ditemukan: "
            + str([str(p) for p in package_candidates])
        )
    if len(summaries) != 1:
        raise RuntimeError(
            "Attach tepat satu coffee17-master-screening-analysis-package.zip "
            "dari master screening yang sudah selesai. "
            f"ZIP={len(package_candidates)}, summaries={len(summaries)}"
        )
    summary_path = summaries[0]
    HIST_ROOT = summary_path.parent.parent if summary_path.parent.name == "analysis" else summary_path.parent
    print("HISTORICAL EXTRACTED ROOT:", HIST_ROOT)

required_history = [
    HIST_ROOT / "analysis" / "master_screening_v1.json",
    *[
        HIST_ROOT / "experiments" / f"fold_{fold}" / "seed42" / "B0" / "run_contract.json"
        for fold in range(1, 6)
    ],
]
missing = [str(p) for p in required_history if not p.is_file()]
if missing:
    raise RuntimeError("Historical master package tidak lengkap: " + str(missing))

hist_summary = json.loads(
    (HIST_ROOT / "analysis" / "master_screening_v1.json").read_text(encoding="utf-8")
)
if hist_summary.get("outer_test_accessed") is not False:
    raise RuntimeError("Historical summary tidak mengunci outer test.")

print("HISTORICAL MASTER COMMIT:", hist_summary.get("git_commit"))
print("HISTORICAL B0 READY FOR AUDIT; IT WILL NOT BE TRAINED.")


In [ ]:
# ------------------------------------------------------------
# Coffee17 dataset: datasets tree only, never notebook-output tree.
# ------------------------------------------------------------
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

DATASETS_INPUT = INPUT / "datasets"
EXPECTED_PUBLIC_ROOT = (
    DATASETS_INPUT
    / "sujitraarw"
    / "coffee-green-bean-with-17-defects-original"
)

if EXPECTED_PUBLIC_ROOT.is_dir():
    DATASET_ROOT = EXPECTED_PUBLIC_ROOT
    by_class = discover_directory_samples(DATASET_ROOT)
else:
    if not DATASETS_INPUT.is_dir():
        raise RuntimeError(f"Kaggle datasets tree tidak ditemukan: {DATASETS_INPUT}")

    valid = []
    roots = [DATASETS_INPUT]
    roots.extend(p for p in sorted(DATASETS_INPUT.rglob("*")) if p.is_dir())
    for root in roots:
        try:
            samples = discover_directory_samples(root)
        except (ValueError, FileNotFoundError):
            continue
        valid.append((root, samples))

    deepest = []
    for root, samples in valid:
        if any(other != root and root in other.parents for other, _ in valid):
            continue
        deepest.append((root, samples))

    if len(deepest) != 1:
        raise RuntimeError(
            "Tidak dapat menentukan satu Coffee17 dataset di /kaggle/input/datasets. "
            f"Kandidat valid: {[str(root) for root, _ in deepest]}"
        )
    DATASET_ROOT, by_class = deepest[0]

raw_count = sum(len(v) for v in by_class.values())
print("USING DATASET INPUT ONLY:", DATASET_ROOT)
print("Coffee17 selected:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        f"Expected Coffee17 original 979/17, observed {raw_count}/{len(by_class)}"
    )

ARCHIVE = WORK / "coffee17_focused_c1c2_original.zip"
PROV = WORK / "coffee17_focused_c1c2_provenance"
CANONICAL = WORK / "coffee17_focused_c1c2_original_v1"
FOLDS = WORK / "coffee17_focused_c1c2_folds"

for path in (PROV, CANONICAL, FOLDS):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV / "coffee17_provenance.json",
    FOLDS,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

print("COFFEE17 PROVENANCE/FOLDS: PASS")


In [ ]:
# ------------------------------------------------------------
# Optional partial-run resume. This is only for C1/C2 checkpoints.
# Historical B0 remains read-only from HIST_ROOT.
# ------------------------------------------------------------
PROJECT.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)

prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-focused-c1c2-v1")
    if p.is_dir()
)
if len(prior_projects) > 1:
    raise RuntimeError(
        "Lebih dari satu focused resume project ditemukan: "
        + str([str(p) for p in prior_projects])
    )
if prior_projects:
    prior = prior_projects[0]
    print("RESTORE PARTIAL C1/C2 PROJECT:", prior)
    if (prior / "experiments").is_dir():
        shutil.copytree(prior / "experiments", OUT, dirs_exist_ok=True)
    if (prior / "logs").is_dir():
        shutil.copytree(prior / "logs", PROJECT / "logs", dirs_exist_ok=True)

CONFIG = REPO / CONFIG_REL

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)

    # Keep the same /kaggle/working path name used by the historical master
    # screening. The runner also matches identities by class/file name.
    DEV = WORK / f"coffee17_master_screen_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)
    materialize_preprocessing_development(
        CANONICAL,
        FOLDS / "clean_manifest.json",
        FOLDS / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_focused_c1c2_v1",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--historical-root", HIST_ROOT,
        "--output-root", OUT,
        "--candidates", "C1", "C2",
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
    ], cwd=REPO, log_path=PROJECT / "logs" / f"fold_{fold}.log")

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

print("\nC1/C2 FIVE-FOLD TRAINING COMPLETE.")
print("B0 TRAINING RUNS IN THIS NOTEBOOK: 0")


In [ ]:
# ------------------------------------------------------------
# Aggregate and compact package
# ------------------------------------------------------------
SUMMARY = PROJECT / "analysis" / "focused_c1c2_v1.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_focused_c1c2_summary_v1",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("\n=== FINAL FOCUSED C1/C2 ===")
for candidate in summary["candidates"]:
    item = summary["aggregate"][candidate]
    print(
        candidate,
        "Macro=", f"{item['metrics']['macro_f1']['mean']:.4%}",
        "dMacro=", f"{item['delta_vs_historical_b0']['macro_f1']['mean']:+.4%}",
        "positive=", f"{item['positive_delta_folds']['macro_f1']}/5",
        "dHard=", f"{item['delta_vs_historical_b0']['hard_class_f1']['mean']:+.4%}",
        "dWorst=", f"{item['delta_vs_historical_b0']['worst_class_f1']['mean']:+.4%}",
        "net=", item["paired_outcomes"]["net_correct"],
    )

STAGE = WORK / "_focused_c1c2_analysis_stage"
ZIP_BASE = WORK / "coffee17-focused-c1c2-analysis-package"
ZIP_PATH = ZIP_BASE.with_suffix(".zip")
shutil.rmtree(STAGE, ignore_errors=True)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
STAGE.mkdir(parents=True)

def copy_file(src: Path, rel: Path):
    if not src.is_file():
        return
    dst = STAGE / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

copy_file(SUMMARY, Path("analysis/focused_c1c2_v1.json"))
copy_file(SUMMARY.with_suffix(".csv"), Path("analysis/focused_c1c2_v1.csv"))
copy_file(
    HIST_ROOT / "analysis" / "master_screening_v1.json",
    Path("historical_master/analysis/master_screening_v1.json"),
)

for fold in range(1, 6):
    fold_root = OUT / f"fold_{fold}" / "seed42"
    copy_file(
        fold_root / "focused_fold_result.json",
        Path("experiments") / f"fold_{fold}" / "seed42" / "focused_fold_result.json",
    )
    for candidate in ("C1", "C2"):
        arm = fold_root / candidate
        rel = Path("experiments") / f"fold_{fold}" / "seed42" / candidate
        for name in ("run_contract.json", "run_config.yaml", "history.json"):
            copy_file(arm / name, rel / name)
        val = arm / "validation"
        if val.is_dir():
            for src in sorted(p for p in val.rglob("*") if p.is_file()):
                copy_file(src, rel / "validation" / src.relative_to(val))

# Keep exact historical B0 compact evidence used by the audit.
for fold in range(1, 6):
    old = HIST_ROOT / "experiments" / f"fold_{fold}" / "seed42" / "B0"
    rel = Path("historical_master") / "experiments" / f"fold_{fold}" / "seed42" / "B0"
    for name in ("run_contract.json", "run_config.yaml"):
        copy_file(old / name, rel / name)
    for name in ("metrics.json", "predictions.csv"):
        copy_file(old / "validation" / name, rel / "validation" / name)

manifest_files = [
    {"path": str(p.relative_to(STAGE)), "bytes": p.stat().st_size}
    for p in sorted(q for q in STAGE.rglob("*") if q.is_file())
]
(STAGE / "PACKAGE_MANIFEST.json").write_text(
    json.dumps(
        {
            "format": "coffee17.focused_c1c2.analysis_package.v1",
            "scientific_code_commit": SCIENTIFIC_CODE_COMMIT,
            "training_candidates": ["C1", "C2"],
            "historical_anchor": "B0",
            "historical_anchor_retrained": False,
            "checkpoints_included": False,
            "outer_test_accessed": False,
            "files": manifest_files,
        },
        indent=2,
    ) + "\n",
    encoding="utf-8",
)

shutil.make_archive(str(ZIP_BASE), "zip", root_dir=STAGE)

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
    if any(name.endswith(".pt") for name in names):
        raise RuntimeError("Analysis package tidak boleh mengandung checkpoint.")
    if not any(name.endswith("focused_c1c2_v1.json") for name in names):
        raise RuntimeError("Focused summary hilang dari ZIP.")

print("\nREADY:", ZIP_PATH)
print("SIZE_MB:", round(ZIP_PATH.stat().st_size / 1024 / 1024, 2))
print("FILES:", len(names))
print("CHECKPOINTS_INCLUDED: 0")

# Cleanup only after the compact ZIP is verified. This prevents another ~GB
# Saved Version while preserving checkpoints if the run aborts before completion.
for pt in PROJECT.rglob("*.pt"):
    pt.unlink()
for path in (REPO, PROV, CANONICAL, FOLDS, HIST_WORK, STAGE):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

print("CLEANUP COMPLETE.")
print("UPLOAD coffee17-focused-c1c2-analysis-package.zip TO CHAT.")
